In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Select features and scale
X = rfm[['log_frequency', 'avg_days_between_orders', 'avg_basket_size']]
X_scaled = StandardScaler().fit_transform(X)

# Elbow + silhouette score
inertias = []
silhouettes = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Plot Elbow
plt.figure()
plt.plot(K, inertias, marker='o')
plt.title('Elbow Method: Inertia vs K')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.show()

# Plot Silhouette
plt.figure()
plt.plot(K, silhouettes, marker='s', color='orange')
plt.title('Silhouette Score vs K')
plt.xlabel('K')
plt.ylabel('Score')
plt.show()


In [ ]:
# Final clustering with selected optimal_k
optimal_k = 4  #selected from the plots
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
rfm['cluster'] = kmeans.fit_predict(X_scaled)

# See how clusters look
rfm.groupby('cluster').agg({
    'frequency': 'mean',
    'avg_days_between_orders': 'mean',
    'avg_basket_size': 'mean',
    'user_id': 'count'
}).rename(columns={'user_id': 'num_customers'})


In [ ]:
# Sample 5000 recent orders
sample_orders = order_products['order_id'].drop_duplicates().sample(5000, random_state=42)
sample_df = order_products[order_products['order_id'].isin(sample_orders)]

# Build basket matrix on this subset
basket = sample_df.groupby(['order_id', 'product_name'])['product_id'].count().unstack().fillna(0)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)
basket.head()

In [ ]:
!pip install mlxtend

In [ ]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)
frequent_itemsets.head(10)

In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules = rules[(rules['confidence'] >= 0.3) & (rules['lift'] >= 1.2)]

rules.sort_values(by='lift', ascending=False).head(10)


In [ ]:
def recommend_next(basket_products, rules_df, top_n=3):
    """
    Given a list of products in a user's basket, return top N recommended products.
    """
    basket_set = set(basket_products)

    # Match rules where basket is a superset of the antecedents
    matched = rules_df[rules_df['antecedents'].apply(lambda x: x.issubset(basket_set))]

    # Sort by lift and confidence
    matched = matched.sort_values(by=['lift', 'confidence'], ascending=False)

    # Collect consequents
    recommended = matched['consequents'].explode().value_counts().head(top_n)
    return list(recommended.index)


In [ ]:
recommend_next(['Organic Avocado', 'Organic Blueberries'], rules)


In [ ]:
!pip install seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare top N rules
top_rules = rules.sort_values(by='lift', ascending=False).head(10).copy()
top_rules['rule'] = top_rules['antecedents'].apply(lambda x: ', '.join(x)) + ' → ' + \
                    top_rules['consequents'].apply(lambda x: ', '.join(x))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='lift', y='rule', data=top_rules, palette='viridis')
plt.title('Top 10 Association Rules by Lift')
plt.xlabel('Lift')
plt.ylabel('Rule')
plt.tight_layout()
plt.show()


In [ ]:
!pip install networkx

In [ ]:
import networkx as nx

# Limiting to top 20 rules to avoid clutter
graph_rules = rules.sort_values(by='lift', ascending=False).head(20)

G = nx.DiGraph()

# Add nodes and edges from rules
for _, row in graph_rules.iterrows():
    for antecedent in row['antecedents']:
        for consequent in row['consequents']:
            G.add_edge(antecedent, consequent, weight=row['lift'], confidence=row['confidence'])

# Draw
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.8)

edges = G.edges()
weights = [G[u][v]['weight'] for u,v in edges]

nx.draw_networkx_nodes(G, pos, node_size=800, node_color='skyblue')
nx.draw_networkx_labels(G, pos, font_size=9)
nx.draw_networkx_edges(G, pos, edgelist=edges, width=2, edge_color=weights, edge_cmap=plt.cm.viridis)

plt.title("Association Rule Network (Top 20 by Lift)")
plt.axis('off')
plt.show()
